In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data/diabetes_prediction_dataset.csv")

In [4]:
df.head(5)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## Shape , datatypes and columns detailes


In [5]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")

Shape: (100000, 9)

Columns:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']

Dtypes:
gender                     str
age                    float64
hypertension             int64
heart_disease            int64
smoking_history            str
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object


In [6]:
print(df['age'].isnull().sum())


0


In [7]:
non_whole_ages = df[df['age'] % 1 != 0]['age']
print(f"fractional ages: {len(non_whole_ages)}")
print(f"Unique fractional values:\n{non_whole_ages.unique()}")

fractional ages: 2018
Unique fractional values:
[0.08 0.56 0.88 0.16 0.64 1.16 1.64 0.72 1.88 1.32 0.8  1.24 1.8  0.48
 1.56 1.08 0.24 1.4  0.4  0.32 1.72 1.48]


In [8]:
infant_rows = df[df['age'] < 2]
print(infant_rows['diabetes'].value_counts())

diabetes
0    2101
Name: count, dtype: int64


In [9]:
print(df['diabetes'].value_counts())
print(f"\nPercentage:\n{df['diabetes'].value_counts(normalize=True) * 100}")

diabetes
0    91500
1     8500
Name: count, dtype: int64

Percentage:
diabetes
0    91.5
1     8.5
Name: proportion, dtype: float64


In [10]:
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Total missing values: 0


In [11]:
print(df['smoking_history'].value_counts())

smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64


In [12]:
df.groupby('smoking_history')['diabetes'].mean() * 100

smoking_history
No Info         4.059638
current        10.208917
ever           11.788212
former         17.001711
never           9.534122
not current    10.702652
Name: diabetes, dtype: float64

In [13]:
df.groupby('smoking_history')['age'].mean()

smoking_history
No Info        33.538037
current        44.063696
ever           49.136863
former         57.061698
never          43.891491
not current    47.689611
Name: age, dtype: float64

In [14]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 3854


In [15]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
47708,Female,2.0,0,0,No Info,27.32,5.0,158,0
59468,Female,2.0,0,0,No Info,27.32,5.0,158,0
62073,Female,2.0,0,0,No Info,27.32,6.0,85,0
67439,Female,2.0,0,0,No Info,27.32,6.0,85,0
23617,Female,2.0,0,0,No Info,27.32,6.0,145,0
67234,Female,2.0,0,0,No Info,27.32,6.0,145,0
90685,Female,2.0,0,0,No Info,27.32,6.2,145,0
97294,Female,2.0,0,0,No Info,27.32,6.2,145,0
46785,Female,2.0,0,0,No Info,27.32,6.5,155,0
89701,Female,2.0,0,0,No Info,27.32,6.5,155,0


In [16]:
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows['age'].describe())
print(f"\nDuplicates with age < 2: {(duplicate_rows['age'] < 2).sum()} out of {len(duplicate_rows)}")

count    6939.000000
mean       43.068413
std        23.152816
min         0.720000
25%        25.000000
50%        42.000000
75%        60.000000
max        80.000000
Name: age, dtype: float64

Duplicates with age < 2: 10 out of 6939


In [17]:
df[['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']].describe()

,age,bmi,HbA1c_level,blood_glucose_level
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,27.320767,5.527507,138.058060
std,22.516840,6.636783,1.070672,40.708136
min,0.080000,10.010000,3.500000,80.000000
25%,24.000000,23.630000,4.800000,100.000000
50%,43.000000,27.320000,5.800000,140.000000
75%,60.000000,29.580000,6.200000,159.000000
max,80.000000,95.690000,9.000000,300.000000


In [18]:
print(df['gender'].value_counts())

gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64


## Data Quality Summary — Diabetes Prediction Dataset

### Issues Identified & Decisions:

1. **Infant/Toddler Records (age < 2)**
   - Finding: 2,101 records with age < 2 years show zero variance in target 
     (100% diabetes = 0), consistent with biological implausibility of 
     type-2 diabetes in this age group.
   - Decision: REMOVE these records.
   - Justification: Domain-driven exclusion; prevents artificial inflation 
     of model accuracy via trivially-separable cases.

2. **Disguised Missing Values — `smoking_history` = "No Info"**
   - Finding: 35,816 records (35.8%) contain "No Info", not detected by 
     standard null checks. Diabetes rate in this group (4.06%) is lower 
     than other categories, but verified to be explained by younger mean 
     age (33.5 yrs) — not a data quality defect.
   - Decision: RETAIN as a distinct category (no imputation, no merging).
   - Justification: Carries genuine demographic signal; imputing would 
     destroy real information.

3. **Duplicate Records**
   - Finding: 3,854 duplicate rows (6,939 total involved), spread across 
     all age groups — attributed to limited precision/discrete nature of 
     features (e.g., bmi to 2 decimals) rather than data entry error.
   - Decision: REMOVE duplicates.
   - Justification: Prevents train-test contamination and inflated 
     apparent performance.

4. **Range/Validity Check — Numeric Columns**
   - Finding: No invalid (zero/negative) values found. `bmi` shows wide 
     range (10.01–95.69) extending into rare clinical extremes.
   - Decision: NO removal at this stage; defer to outlier analysis 
     (Phase 3) for statistical treatment.
   - Justification: Values are medically plausible, not "invalid" — 
     require statistical (not domain) judgment.

5. **Categorical Consistency — `gender` = "Other"**
   - Finding: Only 18 records (0.018%) labeled "Other" — insufficient 
     for the model to learn reliable patterns; risk of zero representation 
     in train/test splits.
   - Decision: REMOVE these 18 records.
   - Justification: Statistical feasibility limitation, not a judgment on 
     category validity. Note as limitation in final report.

### Net Impact:
Starting rows: 100,000
- Remove age < 2: -2,101
- Remove duplicates: -3,854 (approx, some overlap possible with above — 
  to be verified at implementation)
- Remove gender = Other: -18
Estimated final dataset size: ~94,000+ rows (exact count to be confirmed 
during implementation, as removal order may cause overlap)